# 1 Cleaning Data

#### NOTE: We will not load test data here yet to prevent any accidental leakage

In [220]:
!pip3 install scikit-learn
!pip3 install pandas
!pip3 install np

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [221]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv("../dataset/diabetic_data_training.csv")

ids_mapping = pd.read_csv("../dataset/IDS_mapping.csv")

print("training data shape:", df.shape)
print("IDS mapping shape:", ids_mapping.shape)

print("training data")
display(df.head())

print("ids mapping")
display(ids_mapping.head())

training data shape: (91589, 50)
IDS mapping shape: (67, 2)
training data


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
1,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
2,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
3,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO
4,35754,82637451,Caucasian,Male,[50-60),?,2,1,2,3,...,No,Steady,No,No,No,No,No,No,Yes,>30


ids mapping


,admission_type_id,description
0,1,Emergency
1,2,Urgent
2,3,Elective
3,4,Newborn
4,5,Not Available


## 1.1 Checking Target Distribution Balance

#### Note: If target distribution not balanced, for example 90% of people readmitted = "NO", the model might just learn to classify everybody as NO and have 90% accuraccy, which would be very bad

In [222]:
less_than_30 = 0
more_than_30 = 0
not_readmitted = 0
for _, row in df.iterrows():
    if row["readmitted"] == "NO":
        not_readmitted += 1
    elif row["readmitted"] == ">30":
        more_than_30 += 1
    else:
        less_than_30 += 1
print("<30:", less_than_30)
print(">30:", more_than_30)
print("NO:", not_readmitted)

<30: 10245
>30: 31989
NO: 49355


#### This means we an have uneven split of 1 : 3 : 5 and we have moderate class imbalance. This means we must be careful to make a balanced train - validation split. Also, when testing the performance of models, we must make sure to check for recall of the "<30" class to see if the model is catching that.

## 1.2. Train - Validation Split

#### We use the stratify option to ensure that the target variable distributions are the same in the train and val sets. We use the split of 0.2 which is a convention.

In [223]:
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df["readmitted"],random_state=42)

## 1.3. Missing Values

#### Some columns have a lot of values missing. For those, with missing values >70%, we could drop them as they are likely noise (and we can't really fill the values in with median / mode). However, what could potentially better is that we use a binary if that specific thing is relevnt or not (e.g., certain value might only be measured when it is relevant. So waiting for PCA to see if it is actually relevent.)

In [224]:
train_df = train_df.replace('?', np.nan)

missing_column_fraction = (train_df.isnull().sum() / len(train_df))

very_high_missing = missing_column_fraction[missing_column_fraction > 0.7].index.tolist()
high_missing = missing_column_fraction[(missing_column_fraction > 0.2) & (missing_column_fraction <= 0.7)].index.tolist()
low_missing = missing_column_fraction[(missing_column_fraction > 0) & (missing_column_fraction <= 0.2)].index.tolist()
no_missing = missing_column_fraction[missing_column_fraction == 0].index.tolist()

print("Very high missing (>0.7)", very_high_missing)
print("High missing (0.7 > x > 0.2)", high_missing)
print("Low missing (0.2 > x > 0)", low_missing)
print("No missing", no_missing)

Very high missing (>0.7) ['weight', 'max_glu_serum', 'A1Cresult']
High missing (0.7 > x > 0.2) ['payer_code', 'medical_specialty']
Low missing (0.2 > x > 0) ['race', 'diag_1', 'diag_2', 'diag_3']
No missing ['encounter_id', 'patient_nbr', 'gender', 'age', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted']


In [225]:
# For Very High Missing we create a binary column, whether the value has been recorded or not
for col in very_high_missing:
    train_df[col] = 1 - train_df[col].isnull().astype(int)
    train_df = train_df.rename(columns={col: f"is_{col}_recorded"})


# For High Missing we can create a new class of "Unknown", we don't need to create a seperate column to flag this is weird, as it will be
# correlated
for col in high_missing:
    train_df[col] = train_df[col].fillna("Unknown")


# For Low Missing we can reaplce with a median for numerical values, mode for non-numerical. For race, mode is reasonable because there 
# aren't many options, however, for daignoses we could do a new category unknown to not distort the distribution too much. Here we
# need a flag because we tell the model - we guessed that (for race). Fir diag we don't need, same principle as for high_missing
for col in ["diag_1", "diag_2", "diag_3"]:
    train_df[col] = train_df[col].fillna("Unknown")

train_df[f'is_{col}_interpolated'] = train_df[col].isnull().astype(int)
train_df["race"] = train_df["race"].fillna(train_df["race"].mode()[0])


## 1.4. Making sense of the Data

#### We will check whether numerical values should be considered as numerical - if they should be ordered or not

In [226]:
# Since age is actually an ordered column we will take the midpoint value
def age_parser(age_string):
    bounds = age_string.strip("[]()").split("-")
    return (int(bounds[0]) + int(bounds[1])) // 2
train_df["age"] = train_df["age"].apply(lambda x: age_parser(x))


## 1.5. Encoding into Numerical Values

#### Most models need numerical values so for columns with multiple options we can encode as binaries. However, we have to make sure we don't get an explosion of features. Hence we will use top N version of One-Hot-Encoding -> we will only do binary flags for most common N options

In [227]:
column_values = {}
for column in train_df.columns:
    unique_values = set(train_df[column])
    column_values[column] = unique_values
column_values

{'encounter_id': {128057346,
  53870598,
  84934662,
  31850508,
  350879762,
  289669140,
  17563668,
  434372630,
  62783508,
  206831640,
  49545240,
  11665434,
  69337116,
  62259228,
  398458910,
  224002080,
  177995808,
  278265888,
  174194724,
  172621860,
  147718182,
  178126884,
  147980328,
  93978666,
  78643242,
  133693482,
  61866030,
  317325362,
  314966066,
  32899122,
  226623540,
  285868086,
  361365560,
  142868538,
  54263868,
  91881534,
  14417982,
  169214016,
  181141572,
  56492100,
  314966084,
  146931780,
  190447686,
  2228292,
  425459786,
  115605576,
  62521422,
  53870670,
  309592142,
  256376916,
  6291540,
  147325014,
  396099668,
  175112280,
  401342552,
  143655000,
  89522268,
  147718236,
  107610204,
  282198108,
  149946462,
  270794850,
  137887842,
  257687652,
  12058722,
  31064166,
  123994212,
  114294888,
  227016804,
  57409644,
  78512238,
  148897902,
  367919216,
  293470320,
  441450614,
  24117366,
  23986296,
  247988346,


In [228]:
# We first check whether columns are actually consistent - like having a column with 99% integers, 1% unknown or something would be bad

result = []

for col in train_df.columns:
    series = train_df[col]
    total = len(series)
    numeric_count = series.apply(lambda x: isinstance(x, (int, float, np.integer, np.floating)) and not pd.isnull(x)).sum()
    string_count = series.apply(lambda x: isinstance(x, str)).sum()

    result.append({
        'Column': col,
        'Total': total,
        'Numeric': numeric_count,
        'String': string_count,
        'Type Breakdown': f"Strings: {string_count}, Numerics: {numeric_count}"
    })

breakdown_df = pd.DataFrame(result)
print(breakdown_df)


                       Column  Total  Numeric  String  \
0                encounter_id  73271    73271       0   
1                 patient_nbr  73271    73271       0   
2                        race  73271        0   73271   
3                      gender  73271        0   73271   
4                         age  73271    73271       0   
5          is_weight_recorded  73271    73271       0   
6           admission_type_id  73271    73271       0   
7    discharge_disposition_id  73271    73271       0   
8         admission_source_id  73271    73271       0   
9            time_in_hospital  73271    73271       0   
10                 payer_code  73271        0   73271   
11          medical_specialty  73271        0   73271   
12         num_lab_procedures  73271    73271       0   
13             num_procedures  73271    73271       0   
14            num_medications  73271    73271       0   
15          number_outpatient  73271    73271       0   
16           number_emergency  

In [229]:
# Now we see that we have consistent - either all-strings or all-integer columns. Now we can do the top N encoding where necessary.
# Here we don't use the "weird" other binary, because it will be 100% correlated with the group Unknown
# Before choosing this arbitrary number 10, we should first look at how many times the column values repeat
# Actually, better than just choosing the value 10 would be to cpature 90% of the data, 10% is "Other"

for col in train_df.columns:
    if train_df[col].dtype == 'object':
        freq_dict = train_df[col].value_counts().nlargest(500).to_dict()
        if len(freq_dict) > 10:
            print(f"Column: {col}")
            print(freq_dict)
            print("\n")

Column: payer_code
{'Unknown': 28972, 'MC': 23319, 'HM': 4520, 'SP': 3628, 'BC': 3331, 'MD': 2586, 'CP': 1820, 'UN': 1739, 'CM': 1393, 'OG': 743, 'PO': 438, 'DM': 398, 'CH': 110, 'WC': 102, 'OT': 66, 'MP': 61, 'SI': 45}


Column: medical_specialty
{'Unknown': 35890, 'InternalMedicine': 10591, 'Emergency/Trauma': 5418, 'Family/GeneralPractice': 5392, 'Cardiology': 3842, 'Surgery-General': 2261, 'Nephrology': 1159, 'Orthopedics': 1008, 'Orthopedics-Reconstructive': 877, 'Radiologist': 820, 'Pulmonology': 628, 'Psychiatry': 617, 'Urology': 502, 'ObstetricsandGynecology': 498, 'Surgery-Cardiovascular/Thoracic': 458, 'Gastroenterology': 396, 'Surgery-Vascular': 393, 'Surgery-Neuro': 339, 'PhysicalMedicineandRehabilitation': 271, 'Oncology': 248, 'Pediatrics': 191, 'Neurology': 148, 'Hematology/Oncology': 148, 'Pediatrics-Endocrinology': 119, 'Otolaryngology': 92, 'Endocrinology': 83, 'Surgery-Thoracic': 78, 'Podiatry': 74, 'Pediatrics-CriticalCare': 72, 'Psychology': 70, 'Surgery-Cardiovasc

In [230]:
train_df

,encounter_id,patient_nbr,race,gender,age,is_weight_recorded,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted,is_diag_3_interpolated
11895,52900650,23998185,Caucasian,Female,55,0,5,1,17,4,...,No,No,No,No,No,No,No,No,>30,0
43677,147901872,87322464,Caucasian,Female,85,0,5,3,7,1,...,No,No,No,No,No,No,No,No,NO,0
81254,293715854,164631983,AfricanAmerican,Male,55,0,3,1,1,3,...,Up,No,No,No,No,No,Ch,Yes,NO,0
62766,198891540,66366198,Caucasian,Female,65,0,5,3,1,4,...,Up,No,No,No,No,No,Ch,Yes,>30,0
6010,32744148,26052840,Caucasian,Male,85,0,5,1,17,1,...,No,No,No,No,No,No,No,No,NO,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52634,165658218,86119587,Caucasian,Male,65,0,1,1,7,4,...,No,No,No,No,No,No,Ch,Yes,NO,0
16606,66739338,438615,Caucasian,Female,75,0,1,5,7,1,...,No,No,No,No,No,No,No,No,NO,0
41053,140948796,30469320,Caucasian,Male,65,0,1,6,7,11,...,No,No,No,No,No,No,No,No,>30,0
4705,27699294,5217939,AfricanAmerican,Female,55,0,1,1,7,2,...,No,No,No,No,No,No,No,No,NO,0


In [231]:
# Function to take first N so that captures X% of the data
def get_top_n_to_cover_required_fraction(series, coverage=0.91):
    value_counts = series.value_counts()
    cumulative_frac = value_counts.cumsum() / len(series)
    top_categories = cumulative_frac[cumulative_frac <= coverage].index.tolist()
    
    return top_categories


for col in train_df.columns:
    if train_df[col].dtype == 'object' and col != "readmitted":
        if train_df[col].nunique() <= 10:
            top_categories = train_df[col].unique().tolist()
        else: 
            top_categories = get_top_n_to_cover_required_fraction(train_df[col], coverage=0.90)
            
        for value in top_categories:
            value_name = col + "_" + str(value)
            train_df[value_name] = (train_df[col] == value).astype(int)
        
        # If we have some left over, we make the Other column, meaning if we have more than 10
        if len(top_categories) < train_df[col].nunique():
            train_df[f"{col}_Other"] = (~train_df[col].isin(top_categories)).astype(int)

        # Also dropping the original not-numerical column
        train_df = train_df.drop(col, axis=1)

# We have to check if "Other" is even needed - can drop it if not
empty_cols = []
for col in train_df.columns:
    if (train_df[col] == 0).all():
        empty_cols.append(col)

train_df = train_df.drop(columns=empty_cols)  

# Also moving the target variable to the end for structure
cols = [col for col in train_df.columns if col != 'readmitted'] + ['readmitted']
train_df = train_df[cols]

/var/folders/qr/9xd6tq253fd131999hbk5ztc0000gn/T/ipykernel_23412/424779185.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df[value_name] = (train_df[col] == value).astype(int)
/var/folders/qr/9xd6tq253fd131999hbk5ztc0000gn/T/ipykernel_23412/424779185.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df[value_name] = (train_df[col] == value).astype(int)
/var/folders/qr/9xd6tq253fd131999hbk5ztc0000gn/T/ipykernel_23412/424779185.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the res

In [232]:
train_df

,encounter_id,patient_nbr,age,is_weight_recorded,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,...,glimepiride-pioglitazone_No,metformin-rosiglitazone_No,metformin-rosiglitazone_Steady,metformin-pioglitazone_No,metformin-pioglitazone_Steady,change_No,change_Ch,diabetesMed_No,diabetesMed_Yes,readmitted
11895,52900650,23998185,55,0,5,1,17,4,23,0,...,1,1,0,1,0,1,0,1,0,>30
43677,147901872,87322464,85,0,5,3,7,1,40,0,...,1,1,0,1,0,1,0,1,0,NO
81254,293715854,164631983,55,0,3,1,1,3,38,0,...,1,1,0,1,0,0,1,0,1,NO
62766,198891540,66366198,65,0,5,3,1,4,56,1,...,1,1,0,1,0,0,1,0,1,>30
6010,32744148,26052840,85,0,5,1,17,1,27,0,...,1,1,0,1,0,1,0,1,0,NO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52634,165658218,86119587,65,0,1,1,7,4,60,2,...,1,1,0,1,0,0,1,0,1,NO
16606,66739338,438615,75,0,1,5,7,1,60,0,...,1,1,0,1,0,1,0,1,0,NO
41053,140948796,30469320,65,0,1,6,7,11,30,2,...,1,1,0,1,0,1,0,1,0,>30
4705,27699294,5217939,55,0,1,1,7,2,33,2,...,1,1,0,1,0,1,0,1,0,NO


In [20]:
train_df.to_csv("clean_train_df.csv")

In [ ]:
# TO DO:
# use his features explanations... - basically investigate which values are integers intentionally and should be used as such
# Pickle to store the cleaned dataset
# Fix the GitHub things